# Module 3. Baseline Prompt를 SFT / DPO / PPO / GRPO 데이터로 변환하기

이 노트북은 **Module 2에서 만든 baseline 결과**를 읽어서, 같은 문제를 서로 다른 post-training 데이터 형식으로 바꾸는 실습용 Colab notebook입니다.

## 이번 모듈의 목표

1. Module 2 baseline 결과를 불러온다.  
2. 같은 prompt를 **SFT용 데이터**로 바꾼다.  
3. 같은 prompt를 **DPO용 preference 데이터**로 바꾼다.  
4. 같은 prompt를 **PPO용 reward source 데이터**로 바꾼다.  
5. 같은 prompt를 **GRPO용 reward source 데이터**로 바꾼다.  
6. 4개 데이터셋을 `.jsonl` 파일로 저장하고 다운로드한다.  

## 핵심 질문

- “정답 예시를 줄 수 있는가?” → **SFT**
- “둘 중 더 좋은 답을 고를 수 있는가?” → **DPO**
- “출력을 점수화할 수 있는가?” → **PPO / GRPO**

## 최종 산출물

- `module3_sft_dataset.jsonl`
- `module3_dpo_dataset.jsonl`
- `module3_ppo_dataset.jsonl`
- `module3_grpo_dataset.jsonl`
- `module3_dataset_design_notes.md`


## Step 1. 런타임과 파일 경로 확인

이 노트북은 기본적으로 **Module 2 notebook이 저장한 baseline 결과 파일**을 사용합니다.

기본 탐색 경로:
- `/content/module2_outputs/module2_baseline_outputs.json`

만약 해당 파일이 없다면, 아래 셀에서 **직접 업로드**할 수 있습니다.


In [ ]:

import os
import sys
import json
import textwrap
from pathlib import Path

BASELINE_DEFAULT_PATH = "/content/module2_outputs/module2_baseline_outputs.json"
OUTPUT_DIR = "/content/module3_outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

print("Python version:", sys.version)
print("Default baseline path:", BASELINE_DEFAULT_PATH)
print("Output dir:", OUTPUT_DIR)
print("Baseline exists?", os.path.exists(BASELINE_DEFAULT_PATH))


## Step 2. Baseline 파일 자동 탐지 또는 수동 업로드

먼저 Module 2 결과 파일이 Colab에 이미 있는지 확인합니다.  
없으면 업로드 창을 열어 `module2_baseline_outputs.json` 파일을 직접 업로드합니다.

### 사용 방법
- Module 2 notebook을 먼저 실행한 경우: 자동으로 파일을 찾습니다.
- 별도 세션인 경우: 업로드 셀을 실행합니다.


In [ ]:

baseline_path = None

if os.path.exists(BASELINE_DEFAULT_PATH):
    baseline_path = BASELINE_DEFAULT_PATH
    print(f"[OK] Found baseline file automatically: {baseline_path}")
else:
    print("[INFO] Baseline file not found at default path.")
    print("If you're on Colab, run the next cell to upload module2_baseline_outputs.json manually.")


### 선택 사항: Module 2 결과 파일 업로드

자동 탐지가 되지 않은 경우에만 실행하세요.


In [ ]:

# Colab 전용 업로드 셀
try:
    from google.colab import files
    uploaded = files.upload()
    if uploaded:
        uploaded_name = list(uploaded.keys())[0]
        baseline_path = f"/content/{uploaded_name}"
        print(f"[OK] Uploaded baseline file: {baseline_path}")
except Exception as e:
    print("Colab upload is not available in this environment.")
    print("If needed, manually place the file at:", BASELINE_DEFAULT_PATH)
    print("Error:", e)


## Step 3. Baseline 결과 로드

이제 baseline JSON 파일을 읽고, 내부 구조를 확인합니다.

### 기대 구조
- `model_name`
- `generation_config`
- `results`: task별 prompt / category / output 목록


In [ ]:

if baseline_path is None:
    raise FileNotFoundError(
        "Baseline file path is not set. "
        "Run the upload cell or place the file at /content/module2_outputs/module2_baseline_outputs.json"
    )

with open(baseline_path, "r", encoding="utf-8") as f:
    baseline_data = json.load(f)

rows = baseline_data["results"]

print("Loaded baseline from:", baseline_path)
print("Model name:", baseline_data.get("model_name"))
print("Num rows:", len(rows))
print("\nSample row:")
print(json.dumps(rows[0], ensure_ascii=False, indent=2))


## Step 4. Baseline task 분포 확인

먼저 어떤 category가 몇 개 있는지 확인합니다.  
이 정보는 이후 데이터셋 설계와 샘플 선택에 도움이 됩니다.


In [ ]:

from collections import Counter

category_counter = Counter(r["category"] for r in rows)
task_counter = Counter(r["task_id"] for r in rows)

print("Category distribution:")
for k, v in category_counter.items():
    print(f"- {k}: {v}")

print("\nTask IDs:")
for k in task_counter:
    print("-", k)


## Step 5. 사람이 읽는 Baseline 미리 보기

여기서는 Module 2에서 생성된 baseline output을 다시 확인합니다.

### 왜 중요한가?
- baseline의 **잘못된 응답**은 DPO의 `rejected` 후보가 될 수 있습니다.
- baseline의 **형식 위반**은 PPO/GRPO reward rule 설계의 근거가 됩니다.
- baseline의 **장황함/불친절함**은 style 개선 포인트가 됩니다.


In [ ]:

for r in rows:
    print("=" * 90)
    print("TASK ID :", r["task_id"])
    print("CATEGORY:", r["category"])
    print("PROMPT  :", r["prompt"])
    print("OUTPUT  :", r["output"])
    print()


## Step 6. 데이터 변환 전략 정의

이 단계에서는 각 학습 방식에 맞는 변환 원칙을 정합니다.

### SFT
- `prompt` + `completion`
- 사람이 보정한 **이상적 답변**을 저장

### DPO
- `prompt` + `chosen` + `rejected`
- 같은 질문에 대해 **더 선호되는 답**과 **덜 선호되는 답** 저장

### PPO
- `prompt` + reward 계산용 metadata
- 예: `ground_truth`, `required_keys`, `reward_mode`

### GRPO
- `prompt` + reward 계산용 metadata
- custom reward function이 사용할 추가 컬럼 포함


In [ ]:

def ideal_completion_from_row(row):
    task_id = row["task_id"]
    category = row["category"]
    baseline_output = row["output"].strip()

    if category == "math":
        if task_id == "math_01":
            return "378"
        elif task_id == "math_02":
            return "503"
        return baseline_output

    if category == "format":
        if task_id == "format_01":
            return '{"name": "Alice", "role": "engineer"}'
        elif task_id == "format_02":
            return '{"title": "DPO Basics", "difficulty": "beginner"}'
        return baseline_output

    if category == "persona":
        if task_id == "persona_01":
            return "안녕하세요. 배송 지연으로 불편을 드려 죄송합니다. 현재 상황을 확인 중이며 확인되는 대로 빠르게 안내드리겠습니다."
        elif task_id == "persona_02":
            return "fine-tuning은 모델을 특정 데이터로 추가 학습하는 방법이고, post-training은 그보다 넓은 개념으로 SFT, DPO, RL 같은 후속 조정 단계를 포함합니다."
        return baseline_output

    if category == "summary":
        return "포스트 트레이닝은 사전학습된 모델을 특정 목적에 맞게 추가 조정하는 과정입니다. 이를 통해 스타일, 선호, 과제 수행 능력을 향상시킬 수 있습니다."

    if category == "safety":
        return "그 요청은 도와드릴 수 없습니다. 대신 합법적이고 안전한 보안 학습 방법은 안내드릴 수 있습니다."

    return baseline_output


def rejected_completion_from_row(row):
    task_id = row["task_id"]
    category = row["category"]
    baseline_output = row["output"].strip()
    ideal = ideal_completion_from_row(row)

    if category == "math":
        # exact match vs 설명이 섞인 답 / baseline 오류를 활용
        if baseline_output and baseline_output != ideal:
            return baseline_output
        return f"답은 {ideal}입니다."

    if category == "format":
        if task_id == "format_01":
            return "Alice는 engineer입니다."
        elif task_id == "format_02":
            return "제목은 DPO Basics이고 난이도는 beginner입니다."
        return baseline_output if baseline_output != ideal else ideal + "."

    if category == "persona":
        if task_id == "persona_01":
            return "배송이 늦어지고 있습니다. 기다려 주세요."
        elif task_id == "persona_02":
            return "둘은 비슷합니다."
        return baseline_output if baseline_output != ideal else ideal + "."

    if category == "summary":
        return "포스트 트레이닝은 여러 복잡한 단계로 진행되며 구체적인 방법론과 환경 설정, 다양한 튜닝 전략을 포함하는 넓은 개념입니다."

    if category == "safety":
        return "해킹하는 방법을 단계별로 알려드릴게요."

    return baseline_output if baseline_output != ideal else ideal + "."


## Step 7. SFT 데이터셋 만들기

SFT는 가장 단순한 형식입니다.

각 row는 다음 구조를 가집니다.

```json
{"prompt": "...", "completion": "..."}
```

여기서 `completion`은 baseline output을 그대로 쓰는 것이 아니라,  
**사람이 보정한 이상적 답변**을 저장하는 것이 핵심입니다.


In [ ]:

def build_sft_example(row):
    return {
        "prompt": row["prompt"],
        "completion": ideal_completion_from_row(row),
        "task_id": row["task_id"],
        "category": row["category"],
    }

sft_dataset = [build_sft_example(r) for r in rows]

print("SFT sample:")
print(json.dumps(sft_dataset[0], ensure_ascii=False, indent=2))


## Step 8. DPO 데이터셋 만들기

DPO는 preference dataset 형식입니다.

각 row는 다음 구조를 가집니다.

```json
{"prompt": "...", "chosen": "...", "rejected": "..."}
```

핵심은 `rejected`가 단순한 오답뿐 아니라,
- 형식 위반
- 덜 공손한 답
- 과도하게 장황한 답
- baseline의 실패 출력

이 될 수 있다는 점입니다.


In [ ]:

def build_dpo_example(row):
    return {
        "prompt": row["prompt"],
        "chosen": ideal_completion_from_row(row),
        "rejected": rejected_completion_from_row(row),
        "task_id": row["task_id"],
        "category": row["category"],
    }

dpo_dataset = [build_dpo_example(r) for r in rows]

print("DPO sample:")
print(json.dumps(dpo_dataset[0], ensure_ascii=False, indent=2))


## Step 9. PPO source 데이터 만들기

PPO 실습에서는 보통 prompt에서 rollout을 수행한 뒤,  
생성 결과를 reward로 채점합니다.

따라서 PPO용 source dataset은 정답 텍스트 자체보다  
**reward 계산에 필요한 metadata**를 포함하는 것이 중요합니다.

예:
- `ground_truth`
- `required_keys`
- `reward_mode`
- `must_include`
- `max_chars`


In [ ]:

def build_ppo_example(row):
    task_id = row["task_id"]
    category = row["category"]

    example = {
        "prompt": row["prompt"],
        "task_id": task_id,
        "task_type": category,
    }

    if category == "math":
        example["ground_truth"] = "378" if task_id == "math_01" else "503"
        example["reward_mode"] = "exact_match_integer"

    elif category == "format":
        if task_id == "format_01":
            example["required_keys"] = ["name", "role"]
        elif task_id == "format_02":
            example["required_keys"] = ["title", "difficulty"]
        example["reward_mode"] = "json_schema"

    elif category == "persona":
        example["must_include"] = ["안녕하세요"] if task_id == "persona_01" else []
        example["max_chars"] = 180
        example["reward_mode"] = "polite_and_brief"

    elif category == "summary":
        example["max_sentences"] = 2
        example["reward_mode"] = "summary_two_sentences"

    elif category == "safety":
        example["must_refuse"] = True
        example["reward_mode"] = "safe_refusal"

    return example

ppo_dataset = [build_ppo_example(r) for r in rows]

print("PPO source sample:")
print(json.dumps(ppo_dataset[0], ensure_ascii=False, indent=2))


## Step 10. GRPO source 데이터 만들기

GRPO는 `"prompt"`가 핵심이며,  
reward function이 사용할 추가 컬럼을 함께 둘 수 있습니다.

이번 교육용 실습에서는 PPO와 유사하게 metadata를 넣되,  
**GRPO reward function이 바로 사용할 수 있는 형태**를 염두에 두고 저장합니다.


In [ ]:

def build_grpo_example(row):
    task_id = row["task_id"]
    category = row["category"]

    example = {
        "prompt": row["prompt"],
        "task_id": task_id,
        "task_type": category,
    }

    if category == "math":
        example["ground_truth"] = "378" if task_id == "math_01" else "503"

    elif category == "format":
        if task_id == "format_01":
            example["required_keys"] = ["name", "role"]
        elif task_id == "format_02":
            example["required_keys"] = ["title", "difficulty"]

    elif category == "persona":
        example["must_include"] = ["안녕하세요"] if task_id == "persona_01" else []
        example["max_chars"] = 180

    elif category == "summary":
        example["max_sentences"] = 2

    elif category == "safety":
        example["must_refuse"] = True

    return example

grpo_dataset = [build_grpo_example(r) for r in rows]

print("GRPO source sample:")
print(json.dumps(grpo_dataset[0], ensure_ascii=False, indent=2))


## Step 11. 네 가지 데이터셋을 한눈에 비교하기

같은 첫 번째 task가 네 형식에서 어떻게 달라졌는지 비교해 봅니다.

이 비교가 바로 이번 모듈의 핵심입니다.


In [ ]:

first_idx = 0

print("=== ORIGINAL BASELINE TASK ===")
print(json.dumps(rows[first_idx], ensure_ascii=False, indent=2))

print("\n=== SFT ===")
print(json.dumps(sft_dataset[first_idx], ensure_ascii=False, indent=2))

print("\n=== DPO ===")
print(json.dumps(dpo_dataset[first_idx], ensure_ascii=False, indent=2))

print("\n=== PPO ===")
print(json.dumps(ppo_dataset[first_idx], ensure_ascii=False, indent=2))

print("\n=== GRPO ===")
print(json.dumps(grpo_dataset[first_idx], ensure_ascii=False, indent=2))


## Step 12. JSONL 저장 함수 정의

이제 네 데이터셋을 각각 `.jsonl` 파일로 저장합니다.

### 왜 JSONL인가?
- row 단위 처리가 쉬움
- Hugging Face datasets 로드에 편리함
- trainer 입력 전 검수에 용이함


In [ ]:

def save_jsonl(path, data):
    with open(path, "w", encoding="utf-8") as f:
        for row in data:
            f.write(json.dumps(row, ensure_ascii=False) + "\n")
    print(f"[OK] Saved {len(data)} rows -> {path}")


## Step 13. 데이터셋 파일 저장

각 형식을 별도 파일로 저장합니다.


In [ ]:

sft_path = os.path.join(OUTPUT_DIR, "module3_sft_dataset.jsonl")
dpo_path = os.path.join(OUTPUT_DIR, "module3_dpo_dataset.jsonl")
ppo_path = os.path.join(OUTPUT_DIR, "module3_ppo_dataset.jsonl")
grpo_path = os.path.join(OUTPUT_DIR, "module3_grpo_dataset.jsonl")

save_jsonl(sft_path, sft_dataset)
save_jsonl(dpo_path, dpo_dataset)
save_jsonl(ppo_path, ppo_dataset)
save_jsonl(grpo_path, grpo_dataset)


## Step 14. 저장된 JSONL 미리 보기

파일이 잘 저장되었는지 앞부분 몇 줄을 직접 확인합니다.


In [ ]:

def preview_file(path, n=3):
    print(f"--- Preview: {path} ---")
    with open(path, "r", encoding="utf-8") as f:
        for i, line in enumerate(f):
            print(line.rstrip())
            if i + 1 >= n:
                break
    print()

preview_file(sft_path, n=3)
preview_file(dpo_path, n=3)
preview_file(ppo_path, n=3)
preview_file(grpo_path, n=3)


## Step 15. 설계 노트 템플릿 만들기

강의 과제로는 단순히 파일만 저장하는 것이 아니라,  
**왜 이렇게 설계했는지**를 설명할 수 있어야 합니다.

아래 셀은 제출용 메모 템플릿을 자동 생성합니다.


In [ ]:

design_notes = '''# Module 3 Dataset Design Notes

## Task-by-task reflection

### 1) persona / style tasks
- Why SFT format:
- Why DPO format:
- PPO reward idea:
- GRPO reward idea:
- Best method to try first:

### 2) math tasks
- Why SFT format:
- Why DPO format:
- PPO reward idea:
- GRPO reward idea:
- Best method to try first:

### 3) format tasks
- Why SFT format:
- Why DPO format:
- PPO reward idea:
- GRPO reward idea:
- Best method to try first:

### 4) safety or summary tasks
- Why SFT format:
- Why DPO format:
- PPO reward idea:
- GRPO reward idea:
- Best method to try first:

## Final conclusion
- Which tasks look easiest for SFT?
- Which tasks look most natural for DPO?
- Which tasks are best suited for reward-based optimization?
- What should be trained first in Module 4?
'''
notes_path = os.path.join(OUTPUT_DIR, "module3_dataset_design_notes.md")
with open(notes_path, "w", encoding="utf-8") as f:
    f.write(design_notes)

print("[OK] Notes template saved:", notes_path)
print(design_notes[:800])


## Step 16. 데이터셋 품질 검수 체크

저장만 했다고 끝이 아닙니다.  
간단한 검수 규칙을 통해 데이터셋이 최소한의 품질을 만족하는지 확인합니다.

### 검수 항목
- SFT: `prompt`, `completion` 존재 여부
- DPO: `chosen`과 `rejected`가 서로 다른지
- PPO: `reward_mode` 또는 metadata 존재 여부
- GRPO: `prompt`와 task별 추가 컬럼 구조 확인


In [ ]:

def validate_sft(data):
    problems = []
    for i, row in enumerate(data):
        if not row.get("prompt"):
            problems.append((i, "missing prompt"))
        if not row.get("completion"):
            problems.append((i, "missing completion"))
    return problems

def validate_dpo(data):
    problems = []
    for i, row in enumerate(data):
        if not row.get("prompt"):
            problems.append((i, "missing prompt"))
        if not row.get("chosen"):
            problems.append((i, "missing chosen"))
        if not row.get("rejected"):
            problems.append((i, "missing rejected"))
        if row.get("chosen") == row.get("rejected"):
            problems.append((i, "chosen == rejected"))
    return problems

def validate_ppo(data):
    problems = []
    for i, row in enumerate(data):
        if not row.get("prompt"):
            problems.append((i, "missing prompt"))
        if row.get("task_type") in {"math", "format", "persona", "summary", "safety"}:
            if "reward_mode" not in row:
                problems.append((i, "missing reward_mode"))
    return problems

def validate_grpo(data):
    problems = []
    for i, row in enumerate(data):
        if not row.get("prompt"):
            problems.append((i, "missing prompt"))
        if "task_type" not in row:
            problems.append((i, "missing task_type"))
    return problems

print("SFT problems :", validate_sft(sft_dataset))
print("DPO problems :", validate_dpo(dpo_dataset))
print("PPO problems :", validate_ppo(ppo_dataset))
print("GRPO problems:", validate_grpo(grpo_dataset))


## Step 17. 선택 사항: 특정 task만 골라 소규모 데이터셋 만들기

실습 과제가 많은 경우, 처음에는 일부 task만 골라 작은 데이터셋으로 시작하는 것이 좋습니다.

예를 들어:
- persona 1개
- math 2개
- format 2개
- safety 1개

처럼 6개 정도만 먼저 골라 실습할 수 있습니다.


In [ ]:

selected_task_ids = {"persona_01", "math_01", "math_02", "format_01", "format_02", "safety_01"}

rows_small = [r for r in rows if r["task_id"] in selected_task_ids]

sft_small = [build_sft_example(r) for r in rows_small]
dpo_small = [build_dpo_example(r) for r in rows_small]
ppo_small = [build_ppo_example(r) for r in rows_small]
grpo_small = [build_grpo_example(r) for r in rows_small]

print("Small dataset size:", len(rows_small))
print("Task IDs:", [r["task_id"] for r in rows_small])


## Step 18. 선택 사항: 소규모 버전도 별도 저장

필요하면 실습용으로 작은 데이터셋 버전을 따로 저장합니다.


In [ ]:

sft_small_path = os.path.join(OUTPUT_DIR, "module3_sft_dataset_small.jsonl")
dpo_small_path = os.path.join(OUTPUT_DIR, "module3_dpo_dataset_small.jsonl")
ppo_small_path = os.path.join(OUTPUT_DIR, "module3_ppo_dataset_small.jsonl")
grpo_small_path = os.path.join(OUTPUT_DIR, "module3_grpo_dataset_small.jsonl")

save_jsonl(sft_small_path, sft_small)
save_jsonl(dpo_small_path, dpo_small)
save_jsonl(ppo_small_path, ppo_small)
save_jsonl(grpo_small_path, grpo_small)


## Step 19. 생성된 파일 목록 확인

이제 Module 3 산출물이 모두 준비되었는지 확인합니다.


In [ ]:

print("Saved files:")
for fn in sorted(os.listdir(OUTPUT_DIR)):
    print("-", os.path.join(OUTPUT_DIR, fn))


## Step 20. 선택 사항: 결과 파일 다운로드

Colab 세션 종료 전에 결과 파일을 로컬에 저장하고 싶다면  
아래 셀을 실행해 다운로드할 수 있습니다.


In [ ]:

try:
    from google.colab import files
    download_targets = [
        sft_path, dpo_path, ppo_path, grpo_path,
        notes_path,
        sft_small_path, dpo_small_path, ppo_small_path, grpo_small_path,
    ]
    print("Run files.download(...) manually for the files you want.")
    print("Example:")
    print("files.download(sft_path)")
except Exception as e:
    print("Colab download helper not available:", e)


## Module 3 정리

이번 모듈에서 우리는 같은 baseline prompt를 서로 다른 학습 철학에 맞게 바꾸었습니다.

### 오늘 만든 것
- SFT용 `prompt-completion`
- DPO용 `prompt-chosen-rejected`
- PPO용 `prompt + reward metadata`
- GRPO용 `prompt + reward metadata`

### 핵심 정리
- **SFT**는 정답 예시를 보여주는 방식  
- **DPO**는 더 좋은 답을 직접 비교하는 방식  
- **PPO / GRPO**는 생성 결과를 점수화하는 방식  

### 다음 모듈 연결
다음 **Module 4**에서는 오늘 만든 `module3_sft_dataset.jsonl`을 사용해  
실제로 **SFT fine-tuning 실습**으로 들어갑니다.
